## 0. Environment Setup & Imports


In [40]:
import random
import pandas as pd
import joblib

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score



## 1. Vocabulary Loading & Cleaning


In [41]:
with open("correct_words.txt") as f:
    words = set()
    for line in f:
        w = line.strip().lower()
        if w.isalpha() and len(w) >= 3:
            words.add(w)

correct_words = list(words)
print("Total clean correct words:", len(correct_words))



Total clean correct words: 2859


In [42]:
DICTIONARY = set(correct_words)



## 2. Train–Test Split

prevents data leakage.

In [43]:
train_words, test_words = train_test_split(
    correct_words, test_size=0.2, random_state=42
)

print("Train words:", len(train_words))
print("Test words:", len(test_words))


Train words: 2287
Test words: 572


## 3. Synthetic Spelling Error Generation

Synthetic errors simulate common student spelling mistakes.

In [44]:
VOWELS = "aeiou"

def letter_drop(word):
    if len(word) <= 3:
        return None
    i = random.randint(0, len(word)-1)
    return word[:i] + word[i+1:]

def letter_swap(word):
    if len(word) <= 3:
        return None
    i = random.randint(0, len(word)-2)
    return word[:i] + word[i+1] + word[i] + word[i+2:]

def extra_letter(word):
    i = random.randint(0, len(word))
    return word[:i] + random.choice(word) + word[i:]

def vowel_confusion(word):
    chars = list(word)
    idxs = [i for i,c in enumerate(chars) if c in VOWELS]
    if not idxs:
        return None
    i = random.choice(idxs)
    chars[i] = random.choice(VOWELS.replace(chars[i], ""))
    return "".join(chars)



In [45]:
def generate_dataset(words, errors_per_word=3):
    data = []
    generators = [letter_drop, letter_swap, extra_letter, vowel_confusion]

    for word in words:
        # add correct word twice
        data.append((word, 1))
        data.append((word, 1))

        generated = set()
        while len(generated) < errors_per_word:
            gen = random.choice(generators)
            wrong = gen(word)
            if wrong and wrong != word:
                generated.add(wrong)

        for wrong in generated:
            data.append((wrong, 0))

    return data



## 4. Binary Spelling Dataset (Correct vs Incorrect)


In [46]:
train_data = generate_dataset(train_words, errors_per_word=3)
test_data = generate_dataset(test_words, errors_per_word=3)

train_df = pd.DataFrame(train_data, columns=["word", "label"])
test_df = pd.DataFrame(test_data, columns=["word", "label"])

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print(train_df["label"].value_counts())



Train size: 11435
Test size: 2860
label
0    6861
1    4574
Name: count, dtype: int64


In [47]:
vectorizer = CountVectorizer(
    analyzer="char",
   ngram_range=(2,5),   # BETTER for spelling
    min_df=2,
    max_features=30000
)

X_train = vectorizer.fit_transform(train_df["word"])
X_test = vectorizer.transform(test_df["word"])

y_train = train_df["label"]
y_test = test_df["label"]



## 5. Char-Level Spelling Classifier


In [48]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

## 6. Evaluation – Char-Level Model

Macro F1 is used due to class imbalance.

In [49]:
y_pred = (model.predict_proba(X_test)[:,1] > 0.4).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))



Accuracy: 0.7157342657342657
Macro F1: 0.7057431081828595
              precision    recall  f1-score   support

           0       0.77      0.75      0.76      1716
           1       0.64      0.66      0.65      1144

    accuracy                           0.72      2860
   macro avg       0.70      0.71      0.71      2860
weighted avg       0.72      0.72      0.72      2860



In [50]:
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))


Macro F1: 0.7057431081828595


## 7. Error Type Classification

Error classification is applied only after a word is detected as incorrect.



In [51]:
ERROR_LABELS = {
    "correct": 0,
    "letter_drop": 1,
    "letter_swap": 2,
    "extra_letter": 3,
    "vowel_confusion": 4
}


In [52]:
def generate_error_dataset(words, errors_per_word=3):
    data = []

    for word in words:
        # correct
        data.append((word, 0))

        generators = [
            ("letter_drop", letter_drop),
            ("letter_swap", letter_swap),
            ("extra_letter", extra_letter),
            ("vowel_confusion", vowel_confusion)
        ]

        random.shuffle(generators)
        used = set()

        for name, gen in generators:
            if len(used) >= errors_per_word:
                break
            wrong = gen(word)
            if wrong and wrong != word:
                data.append((wrong, ERROR_LABELS[name]))
                used.add(wrong)

    return data


In [53]:
error_data = generate_error_dataset(train_words)

error_df = pd.DataFrame(error_data, columns=["word", "error_type"])

X_err = error_df["word"]
y_err = error_df["error_type"]


In [55]:
error_vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000
)

X_err_vec = error_vectorizer.fit_transform(X_err)


In [56]:
error_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

error_model.fit(X_err_vec, y_err)


LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

## 8. Model Persistence


In [57]:
joblib.dump(model, "spelling_binary_model.pkl")
joblib.dump(vectorizer, "spelling_binary_vectorizer.pkl")

joblib.dump(error_model, "spelling_error_model.pkl")
joblib.dump(error_vectorizer, "spelling_error_vectorizer.pkl")


['spelling_error_vectorizer.pkl']

In [58]:
binary_model = joblib.load("spelling_binary_model.pkl")
binary_vectorizer = joblib.load("spelling_binary_vectorizer.pkl")

error_model = joblib.load("spelling_error_model.pkl")
error_vectorizer = joblib.load("spelling_error_vectorizer.pkl")


## 9. Rule-Based Overrides (Common Misspellings)


In [59]:
COMMON_MISSPELLINGS = {
    "definately": "definitely",
    "seperate": "separate",
    "recieve": "receive",
    "goverment": "government",
    "occured": "occurred",
    "untill": "until",
    "wich": "which"
}


## 10. Char-Level Spell Check Function


In [60]:
ERROR_NAMES = {
    0: "correct",
    1: "letter_drop",
    2: "letter_swap",
    3: "extra_letter",
    4: "vowel_confusion"
}

def check_spelling(word, threshold=0.6):
    word = word.lower()

    # 1️⃣ Hard dictionary check
    if word in DICTIONARY:
        return {
            "word": word,
            "correct": True,
            "confidence": 1.0,
            "error_type": "none"
        }

    # 2️⃣ Common misspelling override
    if word in COMMON_MISSPELLINGS:
        return {
            "word": word,
            "correct": False,
            "confidence": 1.0,
            "error_type": "common_misspelling",
            "suggestion": COMMON_MISSPELLINGS[word]
        }

    # 3️⃣ ML-based plausibility
    X = binary_vectorizer.transform([word])
    prob_correct = binary_model.predict_proba(X)[0, 1]

    if prob_correct > threshold:
        return {
            "word": word,
            "correct": True,
            "confidence": float(prob_correct),
            "error_type": "none"
        }
    else:
        X_err = error_vectorizer.transform([word])
        error_pred = error_model.predict(X_err)[0]

        return {
            "word": word,
            "correct": False,
            "confidence": float(prob_correct),
            "error_type": ERROR_NAMES[error_pred]
        }


## 11. Qualitative Testing – Char-Level Model


In [83]:
test_words = ["hart", "freind", "school", "definately"]

for w in test_words:
    print(check_spelling(w))


{'word': 'hart', 'correct': True, 'confidence': 0.6589575516883234, 'error_type': 'none'}
{'word': 'freind', 'correct': False, 'confidence': 0.10662512701386932, 'error_type': 'letter_swap'}
{'word': 'school', 'correct': True, 'confidence': 1.0, 'error_type': 'none'}
{'word': 'definately', 'correct': False, 'confidence': 1.0, 'error_type': 'common_misspelling', 'suggestion': 'definitely'}


In [62]:
"definately" in DICTIONARY


False

## 12. DistilBERT Plausibility Model


In [13]:
!pip install -q transformers datasets torch accelerate



In [14]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split


In [15]:
VOWELS = "aeiou"

def letter_drop(word):
    if len(word) <= 3: return None
    i = random.randint(0, len(word)-1)
    return word[:i] + word[i+1:]

def letter_swap(word):
    if len(word) <= 3: return None
    i = random.randint(0, len(word)-2)
    return word[:i] + word[i+1] + word[i] + word[i+2:]

def extra_letter(word):
    i = random.randint(0, len(word))
    return word[:i] + random.choice(word) + word[i:]

def vowel_confusion(word):
    chars = list(word)
    idxs = [i for i,c in enumerate(chars) if c in VOWELS]
    if not idxs: return None
    i = random.choice(idxs)
    chars[i] = random.choice(VOWELS.replace(chars[i], ""))
    return "".join(chars)


In [16]:
def generate_bert_dataset(words, errors_per_word=3):
    data = []
    gens = [letter_drop, letter_swap, extra_letter, vowel_confusion]

    for w in words:
        data.append((w, 1))  # correct

        generated = set()
        while len(generated) < errors_per_word:
            wrong = random.choice(gens)(w)
            if wrong and wrong != w:
                generated.add(wrong)

        for wrong in generated:
            data.append((wrong, 0))

    return pd.DataFrame(data, columns=["text", "label"])


In [17]:
with open("correct_words.txt") as f:
    correct_words = [w.strip().lower() for w in f if w.strip().isalpha()]

train_words, test_words = train_test_split(
    correct_words, test_size=0.2, random_state=42
)

train_df = generate_bert_dataset(train_words)
test_df  = generate_bert_dataset(test_words)

print(len(train_df), len(test_df))


9224 2308


In [18]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)


In [19]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=16
    )


In [20]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds  = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")


Map:   0%|          | 0/9224 [00:00<?, ? examples/s]

Map:   0%|          | 0/2308 [00:00<?, ? examples/s]

In [21]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [27]:
training_args = TrainingArguments(
    output_dir="./bert_spellcheck",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)



In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.098700,0.079645,0.974003,0.949580
2,0.069400,0.079010,0.974870,0.950764
3,0.043000,0.089795,0.975737,0.952218


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=867, training_loss=0.08416988225406703, metrics={'train_runtime': 3256.4237, 'train_samples_per_second': 8.498, 'train_steps_per_second': 0.266, 'total_flos': 114551182987776.0, 'train_loss': 0.08416988225406703, 'epoch': 3.0})

## 13. Evaluation – DistilBERT


In [29]:
trainer.evaluate()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.08979488909244537,
 'eval_accuracy': 0.975736568457539,
 'eval_f1': 0.9522184300341296,
 'eval_runtime': 64.3526,
 'eval_samples_per_second': 35.865,
 'eval_steps_per_second': 1.134,
 'epoch': 3.0}

In [82]:

predictions = trainer.predict(test_ds)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["incorrect", "correct"]
    )
)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

   incorrect       0.99      0.98      0.98      1731
     correct       0.94      0.97      0.95       577

    accuracy                           0.98      2308
   macro avg       0.96      0.97      0.97      2308
weighted avg       0.98      0.98      0.98      2308



DistilBERT achieves high recall and F1-score on both correct and incorrect classes, indicating strong performance in estimating spelling plausibility. However, due to its lack of interpretability, it is used as a secondary signal rather than a replacement for the character-level model.

In [30]:
trainer.save_model("distilbert_spelling")
tokenizer.save_pretrained("distilbert_spelling")


('distilbert_spelling/tokenizer_config.json',
 'distilbert_spelling/special_tokens_map.json',
 'distilbert_spelling/vocab.txt',
 'distilbert_spelling/added_tokens.json',
 'distilbert_spelling/tokenizer.json')

In [31]:
from transformers import pipeline

bert_checker = pipeline(
    "text-classification",
    model="distilbert_spelling",
    tokenizer="distilbert_spelling"
)


Device set to use cpu


In [32]:
def bert_score(word):
    out = bert_checker(word)[0]
    return out["score"] if out["label"] == "LABEL_1" else 1 - out["score"]


## 14. Hybrid Inference (DistilBERT + Char-Level ML)


In [76]:
def hybrid_check(word, bert_threshold=0.3, char_threshold=0.6):
    word = word.lower()

    # 1️⃣ Dictionary
    if word in DICTIONARY:
        return {
            "word": word,
            "correct": True,
            "source": "dictionary"
        }

    # 2️⃣ Common misspellings
    if word in COMMON_MISSPELLINGS:
        return {
            "word": word,
            "correct": False,
            "source": "blacklist",
            "suggestion": COMMON_MISSPELLINGS[word]
        }

    # 3️⃣ DistilBERT plausibility
    bert_conf = bert_predict(word)  # probability correct

    # 4️⃣ Char-level ML
    char_result = check_spelling(word, threshold=char_threshold)

    # 5️⃣ Final decision
    if bert_conf < bert_threshold and not char_result["correct"]:
        return {
            **char_result,
            "source": "bert + char"
        }

    if not char_result["correct"]:
        return {
            **char_result,
            "source": "char"
        }

    return {
        "word": word,
        "correct": True,
        "confidence": max(bert_conf, char_result.get("confidence", 0)),
        "source": "bert + char"
    }



In [35]:
SAVE_DIR = "distilbert_spelling_model"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)


('distilbert_spelling_model/tokenizer_config.json',
 'distilbert_spelling_model/special_tokens_map.json',
 'distilbert_spelling_model/vocab.txt',
 'distilbert_spelling_model/added_tokens.json',
 'distilbert_spelling_model/tokenizer.json')

In [36]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_loaded = DistilBertForSequenceClassification.from_pretrained(SAVE_DIR)
tokenizer_loaded = DistilBertTokenizerFast.from_pretrained(SAVE_DIR)


In [37]:
import torch

def bert_predict(word):
    inputs = tokenizer_loaded(
        word,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=16
    )

    with torch.no_grad():
        outputs = model_loaded(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    return probs[0][1].item()  # probability of "correct"


In [73]:
for w in ["booger", "freind", "school", "definately"]:
    print(w, bert_predict(w))


booger 0.0004133391485083848
freind 0.0003251461894251406
school 0.9867709875106812
definately 0.0003308083687443286


## 15. Final End-to-End Tests


In [89]:
for w in ["doctar", "freind", "school", "definately",""]:
    print(w, hybrid_check(w))


doctar {'word': 'doctar', 'correct': False, 'confidence': 0.08699952065443652, 'error_type': 'vowel_confusion', 'source': 'bert + char'}
freind {'word': 'freind', 'correct': False, 'confidence': 0.10662512701386932, 'error_type': 'letter_swap', 'source': 'bert + char'}
school {'word': 'school', 'correct': True, 'source': 'dictionary'}
definately {'word': 'definately', 'correct': False, 'source': 'blacklist', 'suggestion': 'definitely'}
 {'word': '', 'correct': False, 'confidence': 0.5390260616089473, 'error_type': 'letter_drop', 'source': 'char'}


## Conclusion

This notebook demonstrates a hybrid spell-checking system combining:
- Rule-based validation
- Character-level ML for explainability
- DistilBERT for contextual plausibility

This layered design mirrors real-world NLP systems and prioritizes both accuracy and pedagogical feedback.


In [90]:
!zip -r distilbert_spelling.zip distilbert_spelling


  adding: distilbert_spelling/ (stored 0%)
  adding: distilbert_spelling/training_args.bin (deflated 53%)
  adding: distilbert_spelling/model.safetensors (deflated 8%)
  adding: distilbert_spelling/vocab.txt (deflated 53%)
  adding: distilbert_spelling/tokenizer.json (deflated 71%)
  adding: distilbert_spelling/config.json (deflated 45%)
  adding: distilbert_spelling/tokenizer_config.json (deflated 75%)
  adding: distilbert_spelling/special_tokens_map.json (deflated 42%)
